In [1]:
# 1. 安装库
!pip install -q captum shap bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00:00:0100:01


In [2]:
# 2. 配置 Kaggle 路径
import os
from pathlib import Path

# 修改这里：Kaggle 挂载的数据集路径通常是 /kaggle/input/数据集文件夹名
KAGGLE_DATA_PATH = Path("/kaggle/input/datasets/joyeomm/track-a") 
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data root set to: {KAGGLE_DATA_PATH}")

Data root set to: /kaggle/input/datasets/joyeomm/track-a


wrappers

In [3]:
from __future__ import annotations

import math
import warnings
from typing import Callable, Iterable, List, Mapping, Sequence

EmotionVector = List[List[float]]
RawModelOutput = Sequence[float] | Mapping[str, object]

class SentimentWrapper:
    """Unified wrapper that always returns a (batch, 6) emotion probability matrix."""

    labels = ("anger", "disgust", "fear", "joy", "sadness", "surprise")

    def __init__(
        self,
        backend_name: str,
        predictor: Callable[[str], RawModelOutput] | None = None,
    ):
        self.backend_name = backend_name
        self.predictor = predictor or self._heuristic_predictor

    def predict(self, text: str | Sequence[str]) -> EmotionVector:
        if isinstance(text, str):
            texts = [text]
        elif isinstance(text, Sequence):
            texts = list(text)
            if any(not isinstance(item, str) for item in texts):
                raise TypeError("Sequence inputs to SentimentWrapper.predict must contain only strings.")
        else:
            raise TypeError("SentimentWrapper.predict expects a string or a sequence of strings.")
        if not texts:
            return []

        probs_batch: EmotionVector = []
        for item in texts:
            raw = self.predictor(item)
            probs_batch.append(self._to_probabilities(raw))
        return probs_batch

    def _to_probabilities(self, raw: RawModelOutput) -> List[float]:
        if self._is_llama_style(raw):
            return _normalize_to_probability(self._llama_yesno_to_distribution(raw))
        if isinstance(raw, Mapping):
            raise ValueError("Mapping outputs are only supported for llama-style yes/no predictors.")

        values = [float(x) for x in list(raw)]
        if len(values) != len(self.labels):
            raise ValueError(f"Expected {len(self.labels)} outputs, got {len(values)} from {self.backend_name}")

        if any(v < 0.0 or v > 1.0 for v in values):
            values = [_sigmoid(v) for v in values]
        return _normalize_to_probability(values)

    def _is_llama_style(self, raw: RawModelOutput) -> bool:
        if not isinstance(raw, Mapping):
            return False
        if "llama" in self.backend_name.lower():
            return True

        lowered = {str(k).lower(): v for k, v in raw.items()}
        for label in self.labels:
            value = lowered.get(label)
            if isinstance(value, Mapping):
                value_keys = {str(k).lower() for k in value.keys()}
                if "yes" in value_keys or "no" in value_keys or "1" in value_keys or "0" in value_keys:
                    return True
            if isinstance(value, Sequence) and not isinstance(value, (str, bytes)) and len(value) == 2:
                return True
        return False

    def _llama_yesno_to_distribution(self, raw: RawModelOutput) -> List[float]:
        output: List[float] = []
        lowered_map = {str(k).lower(): v for k, v in raw.items()}
        for label in self.labels:
            value = lowered_map.get(label, 0.0)
            output.append(_extract_yes_probability(value))
        return output

    def _heuristic_predictor(self, text: str) -> Sequence[float]:
        lower = text.lower()
        logits = [0.0] * len(self.labels)
        keyword_map = {
            0: ("angry", "mad", "furious", "hate"),
            1: ("disgust", "gross", "nasty", "revolting"),
            2: ("fear", "scared", "afraid", "terrified"),
            3: ("love", "great", "happy", "excellent"),
            4: ("sad", "cry", "down", "depressed"),
            5: ("wow", "surprised", "unexpected", "amazing"),
        }
        for idx, words in keyword_map.items():
            logits[idx] += sum(1.0 for word in words if word in lower)
        return logits

def build_mdeberta_predictor(model_name: str = "microsoft/mdeberta-v3-base") -> Callable[[str], Sequence[float]]:
    try:
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
    except ImportError as exc:
        raise RuntimeError("transformers and torch are required.") from exc

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)
    model.eval()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    def predict_fn(text: str) -> Sequence[float]:
        # 🌟 修复空文本逻辑 (Indentation Fixed)
        if not text.strip():
            return [0.0] * 6 

        encoding = tokenizer(
            text,
            truncation=True,
            max_length=256,
            padding=False,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)
            logits = outputs.logits[0]
            return logits.cpu().numpy().tolist()

    return predict_fn

def build_llama_predictor(
    model_name: str = "meta-llama/Llama-2-7b-hf",
    device_map: str = "auto"
) -> Callable[[str], Mapping[str, object]]:
    try:
        import torch
        import math
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except ImportError as exc:
        raise RuntimeError("transformers and torch are required.") from exc

    print(f"⏳ 正在加载模型并启用极速 Logit 并行优化: {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.padding_side = 'right' # 确保右填充，方便定位 Prompt 结尾
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
        device_map=device_map,
        trust_remote_code=True
    )
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 预先提取 Yes 和 No 在该模型词表里的 Token ID，支持大小写和空格变体
    yes_words = ["Yes", "yes", " Yes", " yes"]
    no_words = ["No", "no", " No", " no"]
    yes_tokens = list(set(tokenizer.encode(w, add_special_tokens=False)[-1] for w in yes_words))
    no_tokens = list(set(tokenizer.encode(w, add_special_tokens=False)[-1] for w in no_words))

    def predict_fn_transformers(text: str) -> Mapping[str, object]:
        emotions = ("anger", "disgust", "fear", "joy", "sadness", "surprise")
        if not text.strip():
            return {emo: {"yes": 0.5, "no": 0.5} for emo in emotions}

        # 🌟 优化 1：将 6 个情绪拼成一个并行 Batch，最大化压榨 GPU 算力
        prompts = [f'Does this text contain {emotion}? Text: "{text}"\nAnswer (Yes/No):' for emotion in emotions]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
        
        # 🌟 优化 2：彻底废除 model.generate()！仅做 1 次正向传播，耗时缩短 95%
        with torch.no_grad():
            outputs = model(**inputs)
        
        results = {}
        for idx, emotion in enumerate(emotions):
            # 通过 attention_mask 动态定位到当前 Prompt 最后一个有效 Token 的位置
            last_token_idx = inputs["attention_mask"][idx].sum().item() - 1
            logits = outputs.logits[idx, last_token_idx, :]
            
            # 提取 Yes 和 No 对应的最大 logit 分数
            yes_logit = max(logits[tid].item() for tid in yes_tokens)
            no_logit = max(logits[tid].item() for tid in no_tokens)
            
            # 运用 Softmax 转化为平滑的概率分布（减去 max_l 防止指数溢出）
            max_l = max(yes_logit, no_logit)
            exp_yes = math.exp(yes_logit - max_l)
            exp_no = math.exp(no_logit - max_l)
            yes_prob = exp_yes / (exp_yes + exp_no)
            
            results[emotion] = {"yes": yes_prob, "no": 1.0 - yes_prob}
            
        return results

    return predict_fn_transformers
   

def _extract_yes_probability(value: object) -> float:
    if isinstance(value, Mapping):
        lowered = {str(k).lower(): v for k, v in value.items()}
        yes = float(lowered.get("yes", lowered.get("1", 0.0)))
        no = float(lowered.get("no", lowered.get("0", 1.0 - yes)))
        denom = yes + no
        return yes / denom if denom > 0 else 0.0
    if isinstance(value, Sequence) and not isinstance(value, (str, bytes)):
        arr = list(value)
        if len(arr) == 2:
            denom = float(arr[0]) + float(arr[1])
            return float(arr[0]) / denom if denom > 0 else 0.0
    return float(value)

def _sigmoid(x: float) -> float:
    if x >= 0:
        return 1.0 / (1.0 + math.exp(-x))
    z = math.exp(x)
    return z / (1.0 + z)

def _normalize_to_probability(raw: Iterable[float]) -> List[float]:
    values = [float(x) for x in raw]
    min_v = min(values)
    if min_v < 0:
        values = [v - min_v for v in values]
    total = sum(values)
    if total == 0:
        return [1.0 / len(values)] * len(values)
    probs = [v / total for v in values]
    if not math.isclose(sum(probs), 1.0):
        probs[-1] = 1.0 - sum(probs[:-1])
    return probs

metrics

In [9]:
from __future__ import annotations

from typing import Iterable, List


def _as_curve(probability_curve: Iterable[float]) -> List[float]:
    return [float(x) for x in probability_curve]


def aopc(probability_curve: Iterable[float]) -> float:
    """Area Over the Perturbation Curve using average confidence drop from p0."""
    curve = _as_curve(probability_curve)
    if len(curve) < 2:
        return 0.0

    p0 = curve[0]
    drops = [max(0.0, p0 - pk) for pk in curve[1:]]
    return sum(drops) / len(drops)


def naopc(probability_curve: Iterable[float], reference_probability: float | None = None) -> float:
    """Normalized AOPC in [0,1] for cross-model/language comparison.

    Defaults to using the final confidence after full perturbation as the
    normalization reference when no explicit reference_probability is provided.
    """
    curve = _as_curve(probability_curve)
    if len(curve) < 2:
        return 0.0

    eps = 1e-12
    denom_ref = float(reference_probability) if reference_probability is not None else curve[-1]
    normalizer = curve[0] - denom_ref
    if normalizer <= eps:
        return 0.0

    score = aopc(curve) / normalizer
    return max(0.0, min(1.0, score))

explainers

In [10]:
from __future__ import annotations

import warnings
from typing import Callable, List, Sequence, Tuple



TokenCandidates = Sequence[Tuple[str, float]]
MaskedPredictor = Callable[[str, str, int], TokenCandidates]


def _tokenize(text: str) -> List[str]:
    return text.split()


def _untokenize(tokens: Sequence[str]) -> str:
    return " ".join(tokens).strip()


def loo_importance(wrapper: SentimentWrapper, text: str, label_idx: int) -> List[float]:
    tokens = _tokenize(text)
    if not tokens:
        return []

    base = wrapper.predict(text)[0][label_idx]
    scores: List[float] = []

    for i in range(len(tokens)):
        perturbed = _untokenize(tokens[:i] + tokens[i + 1 :])
        perturbed_prob = wrapper.predict(perturbed)[0][label_idx]
        scores.append(base - perturbed_prob)
    return scores


def build_mlm_masked_predictor(model_name: str = "bert-base-multilingual-cased") -> MaskedPredictor:
    """Create a masked-token predictor used by input marginalization.

    Implements p(w|context) with a multilingual masked-LM.
    """

    try:
        import torch
        from transformers import AutoModelForMaskedLM, AutoTokenizer
    except ImportError as exc:  # pragma: no cover - optional dependency fallback
        raise RuntimeError(
            "transformers and torch are required for MLM marginalization predictor "
            "(install with: pip install transformers torch)."
        ) from exc

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    model.eval()

    if tokenizer.mask_token is None or tokenizer.mask_token_id is None:
        raise ValueError(f"Model '{model_name}' does not define a mask token.")

    def predict(prefix: str, suffix: str, k: int) -> TokenCandidates:
        masked_text = _untokenize([piece for piece in (prefix, tokenizer.mask_token, suffix) if piece])
        encoded = tokenizer(masked_text, return_tensors="pt")
        input_ids = encoded["input_ids"][0]
        mask_positions = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=False)
        if mask_positions.numel() == 0:
            return []

        mask_pos = int(mask_positions[0].item())
        with torch.no_grad():
            logits = model(**encoded).logits[0, mask_pos]
            probs = torch.softmax(logits, dim=-1)

        top_k = max(1, min(int(k), probs.shape[-1]))
        values, indices = torch.topk(probs, k=top_k)

        candidates: List[Tuple[str, float]] = []
        for token_id, prob in zip(indices.tolist(), values.tolist()):
            token = tokenizer.convert_ids_to_tokens([int(token_id)])[0]
            word = tokenizer.convert_tokens_to_string([token]).strip()
            if not word:
                continue
            candidates.append((word, float(prob)))
        return candidates

    return predict


def marginalization_importance(
    wrapper: SentimentWrapper,
    text: str,
    label_idx: int,
    masked_predictor: MaskedPredictor,
    top_k: int = 5,
) -> List[float]:
    """Input marginalization attribution following Eq. p(y|x\\w_i)=sum_w p(w|ctx)p(y|x[w_i->w])."""
    tokens = _tokenize(text)
    if not tokens:
        return []

    base = wrapper.predict(text)[0][label_idx]
    scores: List[float] = []

    for i in range(len(tokens)):
        prefix = _untokenize(tokens[:i])
        suffix = _untokenize(tokens[i + 1 :])
        candidates = masked_predictor(prefix, suffix, top_k)
        if not candidates:
            scores.append(0.0)
            continue

        expected_prob = 0.0
        total_weight = 0.0
        for replacement, weight in candidates:
            if weight <= 0:
                continue
            replaced = tokens[:i] + [replacement] + tokens[i + 1 :]
            expected_prob += float(weight) * wrapper.predict(_untokenize(replaced))[0][label_idx]
            total_weight += float(weight)

        if total_weight <= 0:
            scores.append(0.0)
            continue
        expected_prob /= total_weight
        scores.append(base - expected_prob)

    return scores


def lime_importance(wrapper: SentimentWrapper, text: str, label_idx: int) -> List[float]:
    """LIME integration with graceful fallback to LOO if lime is unavailable."""
    tokens = _tokenize(text)
    if not tokens:
        return []

    try:
        from lime.lime_text import LimeTextExplainer
    except ImportError:
        return loo_importance(wrapper, text, label_idx)

    explainer = LimeTextExplainer(class_names=list(wrapper.labels), split_expression=r"\s+")

    def classifier_fn(texts: Sequence[str]) -> list[list[float]]:
        return wrapper.predict(list(texts))

    try:
        explanation = explainer.explain_instance(
            text,
            classifier_fn,
            labels=(label_idx,),
            num_features=len(tokens),
        )
        weights = dict(explanation.as_list(label=label_idx))
        return [float(weights.get(tok, 0.0)) for tok in tokens]
    except (ValueError, KeyError, TypeError) as exc:  # pragma: no cover - optional backend behavior
        warnings.warn(f"LIME failed ({exc!r}); falling back to LOO scores.", RuntimeWarning, stacklevel=2)
        return loo_importance(wrapper, text, label_idx)


def shap_importance(wrapper: SentimentWrapper, text: str, label_idx: int) -> List[float]:
    """SHAP integration with graceful fallback to LOO if shap is unavailable."""
    tokens = _tokenize(text)
    if not tokens:
        return []

    try:
        import shap
    except ImportError:
        return loo_importance(wrapper, text, label_idx)

    def model_fn(texts: Sequence[str]) -> list[list[float]]:
        return wrapper.predict(list(texts))

    try:
        masker = shap.maskers.Text(r"\W+")
        explainer = shap.Explainer(model_fn, masker)
        values = explainer([text])
        token_values = values.values[0]
        if token_values.ndim == 2:
            class_values = token_values[:, label_idx]
        else:
            class_values = token_values
        class_values_list = [float(v) for v in class_values]
        if len(class_values_list) == len(tokens):
            return class_values_list
    except (ValueError, KeyError, TypeError) as exc:  # pragma: no cover - optional backend behavior
        warnings.warn(f"SHAP failed ({exc!r}); falling back to LOO scores.", RuntimeWarning, stacklevel=2)

    return loo_importance(wrapper, text, label_idx)


def plex_importance(
    wrapper: SentimentWrapper,
    text: str,
    label_idx: int,
    masked_predictor: MaskedPredictor,
    top_k: int = 3,
) -> List[float]:
    """Perturbation-light proxy: equal-weight blend of LOO and low-k marginalization for lower query cost."""
    loo_scores = loo_importance(wrapper, text, label_idx)
    marg_scores = marginalization_importance(wrapper, text, label_idx, masked_predictor=masked_predictor, top_k=top_k)
    if not loo_scores:
        return marg_scores
    if len(loo_scores) != len(marg_scores):
        warnings.warn("PLEX score-length mismatch detected; returning LOO scores.", RuntimeWarning, stacklevel=2)
        return loo_scores
    return [(l + m) / 2.0 for l, m in zip(loo_scores, marg_scores)]

run core 3 experiment

In [11]:
from __future__ import annotations

import argparse
import json
import random
import time  # 🌟 新增：用于精准统计每种方法的耗时
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, List, Sequence

import pandas as pd

# 保持你原有的辅助工具函数不变
def _default_masked_predictor(prefix: str, suffix: str, _k: int) -> list[tuple[str, float]]:
    del prefix, suffix
    return [("the", 1.0)]

def _remove_top_k_tokens(text: str, scores: Sequence[float], k: int) -> str:
    tokens = text.split()
    if not tokens:
        return text
    k = max(0, min(k, len(tokens)))
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    remove = set(ranked[:k])
    return " ".join(tok for i, tok in enumerate(tokens) if i not in remove)

def _curve_from_scores(wrapper: SentimentWrapper, text: str, label_idx: int, scores: Sequence[float]) -> List[float]:
    tokens = text.split()
    if not tokens:
        return [wrapper.predict(text)[0][label_idx]]

    curve = [wrapper.predict(text)[0][label_idx]]
    for k in range(1, len(tokens) + 1):
        perturbed = _remove_top_k_tokens(text, scores, k)
        curve.append(wrapper.predict(perturbed)[0][label_idx])
    return curve

def _sample_rows(df: pd.DataFrame, sample_size: int, seed: int) -> pd.DataFrame:
    if len(df) <= sample_size:
        return df.copy()
    rng = random.Random(seed)
    indices = rng.sample(list(df.index), sample_size)
    return df.loc[indices].reset_index(drop=True)

# 🌟 全新重构：支持分语种动态调度、单方法计时、高频自动刷盘的流水线
def run_pipeline_dynamic(
    data_root: Path,
    output_dir: Path,
    lang: str,                  # 修改：每次专门处理一个语种，控制粒度更细
    sample_size: int,
    seed: int,
    use_mlm: bool,
    mlm_model: str,
    wrapper: SentimentWrapper  
) -> pd.DataFrame:
    
    # 初始化 MLM (Marginalization 的核心)
    masked_predictor = _default_masked_predictor
    if use_mlm:
        print(f"  [Assistant] 正在同步加载解释助手模型: {mlm_model}...")
        masked_predictor = build_mlm_masked_predictor(mlm_model)

    records = []
    
    # 路径自适应：兼容直接在根目录或者在语言子目录下的情况
    path = data_root / f"{lang}.csv"
    if not path.exists():
        path = data_root / lang / f"{lang}.csv"
        if not path.exists():
            raise FileNotFoundError(f"找不到数据文件。已尝试路径: {data_root / f'{lang}.csv'} 和 {data_root / lang / f'{lang}.csv'}")

    print(f"\n🚀 [核心启动] 当前语种: {lang.upper()} | 目标采样数: {sample_size}")
    df = pd.read_csv(path)
    sampled = _sample_rows(df, sample_size=sample_size, seed=seed)
    
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_file = output_dir / f"core3_{lang}_results.csv" # 每一个语言拥有完全独立的物理文件

    for i, (row_idx, row) in enumerate(sampled.iterrows()):
        text = str(row["text"])
        if not text.strip():
            continue
            
        base_probs = wrapper.predict(text)[0]
        label_idx = int(max(enumerate(base_probs), key=lambda item: item[1])[0])

        # 将 5 种方法拆解开，方便单独套用计时器和异常防护网
        methods = {
            "loo": lambda: loo_importance(wrapper, text, label_idx),
            "shap": lambda: shap_importance(wrapper, text, label_idx),
            "lime": lambda: lime_importance(wrapper, text, label_idx),
            "marginalization": lambda: marginalization_importance(wrapper, text, label_idx, masked_predictor=masked_predictor),
            "plex": lambda: plex_importance(wrapper, text, label_idx, masked_predictor=masked_predictor),
        }

        for method_name, method_fn in methods.items():
            start_time = time.time()  # ⏱️ 启动计时
            try:
                scores = method_fn()
                curve = _curve_from_scores(wrapper, text, label_idx, scores)
                naopc_score = naopc(curve)
                elapsed_time = time.time() - start_time  # ⏱️ 结束计时
                
                records.append({
                    "language": lang,
                    "row_index": int(row_idx),
                    "method": method_name,
                    "label_idx": label_idx,
                    "token_count": len(text.split()),
                    "naopc": naopc_score,
                    "elapsed_time_seconds": round(elapsed_time, 4), # 🌟 新增：独立记录该方法在此样本上的耗时
                    "text_snippet": text[:30] + "..."
                })
            except Exception as method_err:
                # 方法级隔离：某一种方法（如 LIME）报错崩溃，不影响别的解释方法计算，更不会中断整个大循环
                print(f"    ⚠️ 样本 {i+1} 运行 {method_name} 方法时跳过 (异常捕获): {method_err}")
                continue

        # 🌟 超高频安全写盘：每处理 5 条数据，强行把已有结果存入硬盘。断网下班也绝不丢失进度！
        if (i + 1) % 5 == 0 or (i + 1) == len(sampled):
            current_df = pd.DataFrame(records)
            current_df.to_csv(checkpoint_file, index=False)
            print(f"  📸 [自动刷盘] {lang.upper()} 已安全存盘 {i+1}/{sample_size} 个样本...")

    return pd.DataFrame(records)

In [ ]:
import pandas as pd
from pathlib import Path

# --- 配置与路径（直接复用你 Cell 1 的全局变量） ---
mdeberta_results_path = OUTPUT_DIR / "mdeberta_results"
mdeberta_results_path.mkdir(exist_ok=True)

# 🌟 根据你 Kaggle 数据集 test/ 目录下的真实文件名修改（比如 "eng", "afr", "jav"）
target_languages = ["eng", "afr", "jav"] 

# 1. 初始化模型 (放在循环外，只加载一次，最大化压榨 GPU 效率)
print("⏳ 正在将主模型 microsoft/mdeberta-v3-base 加载至 GPU 显存...")
predictor_m = build_mdeberta_predictor("microsoft/mdeberta-v3-base")
wrapper_m = SentimentWrapper("mdeberta", predictor_m)

# 2. 分语言运行实验
all_lang_results = {}

for lang in target_languages:
    print(f"\n{'='*40}")
    print(f" 🎬 启动语种大循环: {lang.upper()}")
    print(f"{'='*40}")
    
    # 为每种语言创建单独的结果子目录
    lang_output_dir = mdeberta_results_path / lang
    lang_output_dir.mkdir(exist_ok=True)
    
    try:
        # 调用上面重构的全新 pipeline 函数
        results_lang_df = run_pipeline_dynamic(
            data_root=KAGGLE_DATA_PATH / "test", # 确保指向测试集根路径
            output_dir=lang_output_dir,
            lang=lang,
            sample_size=50,  # 调整为 100 样本，兼顾因果分析质量与挂机时间
            seed=42,
            use_mlm=True,
            mlm_model="bert-base-multilingual-cased",
            wrapper=wrapper_m 
        )
        
        # 语种完成后的双重归档备份
        summary_file = lang_output_dir / f"summary_{lang}_final.csv"
        results_lang_df.to_csv(summary_file, index=False)
        
        all_lang_results[lang] = results_lang_df
        print(f"✅ 【阶段性胜利】 {lang.upper()} 已经全部稳妥闭环！最终文件已保存。")
        
    except Exception as lang_err:
        # 语种级隔离保护：即使某个语种路径配错或崩溃，也会自动跳向下一个，不让你之前的挂机白费
        print(f"❌ 【严重阻断】 语种 {lang.upper()} 发生致命错误: {lang_err}")
        print("  程序将自动执行下一个语种...")
        continue 

print("\n🎉 🎉 🎉 【全剧终】所有指定语种的 XAI 实验全部安全收官！")

In [ ]:
!zip -r hazards_results.zip /kaggle/working
from IPython.display import FileLink
FileLink(r'hazards_results.zip')


In [ ]:
# --- 启动极速 Logit 版 0.5B 大模型实验 ---

print("⏳ 正在初始化 Qwen2.5-0.5B 情感预测包装器（Logit 加速版）...")
predictor_q = build_llama_predictor("Qwen/Qwen2.5-0.5B-Instruct") 
wrapper_q = SentimentWrapper("llama", predictor_q)

# 启动高频刷盘实验
results_qwen_df = run_pipeline_dynamic(
    data_root=KAGGLE_DATA_PATH / "test",
    output_dir=OUTPUT_DIR / "qwen_results",
    lang="eng",  
    sample_size=20, # 30个样本预计几分钟就能全部冲完
    seed=42,
    use_mlm=True,
    mlm_model="bert-base-multilingual-cased",
    wrapper=wrapper_q
)

In [12]:
# --- 启动极速 Logit 版 0.5B 大模型实验 ---

print("⏳ 正在初始化 Qwen2.5-0.5B 情感预测包装器（Logit 加速版）...")
predictor_q = build_llama_predictor("Qwen/Qwen2.5-0.5B-Instruct") 
wrapper_q = SentimentWrapper("llama", predictor_q)

# 启动高频刷盘实验
results_qwen_df = run_pipeline_dynamic(
    data_root=KAGGLE_DATA_PATH / "test",
    output_dir=OUTPUT_DIR / "qwen_results",
    lang="afr",  
    sample_size=20, # 30个样本预计几分钟就能全部冲完
    seed=42,
    use_mlm=True,
    mlm_model="bert-base-multilingual-cased",
    wrapper=wrapper_q
)

⏳ 正在初始化 Qwen2.5-0.5B 情感预测包装器（Logit 加速版）...
⏳ 正在加载模型并启用极速 Logit 并行优化: Qwen/Qwen2.5-0.5B-Instruct ...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

PartitionExplainer explainer: 2it [00:11, 11.87s/it]               


  0%|          | 0/132 [00:00<?, ?it/s]

  0%|          | 0/380 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:15, 15.72s/it]               


  📸 [自动刷盘] AFR 已安全存盘 20/20 个样本...


In [13]:
# --- 启动极速 Logit 版 0.5B 大模型实验 ---

print("⏳ 正在初始化 Qwen2.5-0.5B 情感预测包装器（Logit 加速版）...")
predictor_q = build_llama_predictor("Qwen/Qwen2.5-0.5B-Instruct") 
wrapper_q = SentimentWrapper("llama", predictor_q)

# 启动高频刷盘实验
results_qwen_df = run_pipeline_dynamic(
    data_root=KAGGLE_DATA_PATH / "test",
    output_dir=OUTPUT_DIR / "qwen_results",
    lang="jav",  
    sample_size=20, # 30个样本预计几分钟就能全部冲完
    seed=42,
    use_mlm=True,
    mlm_model="bert-base-multilingual-cased",
    wrapper=wrapper_q
)

⏳ 正在初始化 Qwen2.5-0.5B 情感预测包装器（Logit 加速版）...
⏳ 正在加载模型并启用极速 Logit 并行优化: Qwen/Qwen2.5-0.5B-Instruct ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  [Assistant] 正在同步加载解释助手模型: bert-base-multilingual-cased...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

  0%|          | 0/210 [00:00<?, ?it/s]

  0%|          | 0/306 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:21, 21.45s/it]               


  📸 [自动刷盘] JAV 已安全存盘 20/20 个样本...
